## Structured Output

Models can be requested to provide their response in a format matching a given Schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing strcutured output

#### Pydantic

In [2]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001586BDBEC10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001586B917210>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(
        json_schema_extra={"description": "The title of the movie"}
    )
    year: int = Field(
        json_schema_extra={"description": "The year the movie was released"}
    )
    director: str = Field(
        json_schema_extra={"description": "The director of the movie"}
    )
    rating: float = Field(
        json_schema_extra={"description": "The movie rating out of 10"}
    )

In [5]:

llm_with_structure=llm.with_structured_output(Movie)
llm_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001586BDBEC10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001586B917210>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movie rating out of 10', 'type': 'number'}}, 'required': ['title', '

In [6]:
llm.invoke("Provide the detals about the movie inception")

AIMessage(content='**Inception (2010)**\n\nInception is a mind-bending science fiction action film written, co-produced, and directed by Christopher Nolan. The film features an ensemble cast, including Leonardo DiCaprio, Joseph Gordon-Levitt, Ellen Page, Tom Hardy, Ken Watanabe, Dileep Rao, Cillian Murphy, Tom Berenger, and Marion Cotillard.\n\n**Plot**\n\nThe movie follows Cobb (Leonardo DiCaprio), a skilled thief who specializes in entering people\'s dreams and stealing their secrets. Cobb is hired by a wealthy businessman named Saito (Ken Watanabe) to perform a task known as "inception" - planting an idea in someone\'s mind instead of stealing one.\n\nSaito wants Cobb to convince Robert Fischer (Cillian Murphy), the son of a dying business magnate, to dissolve his father\'s company. In return, Saito promises to clear Cobb\'s name, which is wanted by the authorities, and allow him to return to the United States to see his children.\n\nCobb assembles a team of experts to help him perf

In [7]:
response = llm_with_structure.invoke("Provide the details about the movie Inception")

In [8]:
print(response)

title='Inception' year=2010 director='Christopher Nolan' rating=8.5


### message output alongside parsed Structure

In [9]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(
        json_schema_extra={"description": "The title of the movie"}
    )
    year: int = Field(
        json_schema_extra={"description": "The year the movie was released"}
    )
    director: str = Field(
        json_schema_extra={"description": "The director of the movie"}
    )
    rating: float = Field(
        json_schema_extra={"description": "The movie rating out of 10"}
    )

In [10]:
llm_with_structure=llm.with_structured_output(Movie, include_raw=True)

In [11]:
response = llm_with_structure.invoke("Provie me details about movie iron man 1")

In [12]:
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'wcmgzx3wj', 'function': {'arguments': '{"director":"Jon Favreau","rating":7.9,"title":"Iron Man","year":2008}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 282, 'total_tokens': 315, 'completion_time': 0.087459465, 'completion_tokens_details': None, 'prompt_time': 0.051896574, 'prompt_tokens_details': None, 'queue_time': 0.157116415, 'total_time': 0.139356039}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d66ba-14b5-7740-a6ae-a50148a68ce7-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Jon Favreau', 'rating': 7.9, 'title': 'Iron Man', 'year': 2008}, 'id': 'wcmgzx3wj', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 282, 'output_tokens': 33, 'total_tokens

#### Nested Structure

In [13]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str
    
    
class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None, json_schema_extra={"description":"Budget in million USD"})

In [14]:
model_with_structure = llm.with_structured_output(MovieDetails)


In [15]:
response = model_with_structure.invoke("Provide me the details about Dhurandhar: The Revenge")

print(response)

title='Dhurandhar: The Revenge' year=2023 cast=[Actor(name='Kalyan Ram', role='Dhurandhar'), Actor(name='Arjun Rampal', role='Rudra'), Actor(name='Ashika Ranganath', role='Anvi')] genres=['Action', 'Thriller'] budget=15.0


### TypedDict

TypedDict provides a simpler alternative using pythons built-in typing , ideal when you dont need runtime validation 

In [16]:
from typing_extensions import TypedDict,Annotated

In [17]:
class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str, ...,"The title of the movie"]
    year: Annotated[int, ...,"The year movie was released"]
    director: Annotated[str, ...,"The director of the movie"]
    rating: Annotated[float, ...,"The movie rating out of 10"]
    

In [18]:
model_withtypedDict = llm.with_structured_output(MovieDict)

In [19]:
response = model_withtypedDict.invoke("please provide me the details about the movie Avengers endgame")

In [20]:
print(response)

{'director': 'Anthony Russo', 'rating': 8.4, 'title': 'Avengers: Endgame', 'year': 2019}


In [ ]:
#